# Predicting Protein Prediction

Consider as starting point for this exercise a UCI Protein Structure. The dataset comes from the Critical Assessment of protein Structure Prediction experiments (CASP), which is a recurrent (biannual) initiative to predict protein structure from experimental data.

The dataset consists of roughly 45k entries with nine features and one target. 

The features essentially are calculated physicochemical descriptors:
- F1: Total surface area (Approximate exposed surface of the protein)
- F2: Non-polar exposed area (Hydrophobic surface)
- F3: Fraction of exposed nonpolar area (Ratio of hydrophobic and total surface)
- F4: Residue surface exposure (How much amino acids are exposed)
- F5: Secondary structure agreement (Measures consistency with expected structures (α-helices, β-sheets))
- F6: Pairwise distance features (Encodes distances between residues)
- F7: Compactness / packing (How tightly folded the protein is)
- F8: Structural energy-related feature (Proxy for physical plausibility)
- F9: Additional geometric descriptor (Captures global structure properties)

The target is the RMSD (Root Mean Squared Deviation) that describes the deviation of the predicted from the true protein structure. 

The aim of the exercise is to build a model to predict how accurate predicted structures would be based on calculated descriptors.

#### Tasks:
1) The data is somewhat abstract. Inspect it to see what can be expected of a potential model.
2) Create feature matrix and target vector.
3) Choose one Regression ML model, build it and optimise (consider scaling if the model class needs it)
4) Take note of the training and test time for your model (approximation is enough)
5) Whatever model you end up using, try to optimise for accuracy and minimal overfitting, use **MSE** for evaluating your model!
6) Respond to the discussion points.

#### Note:
Feel free in your choice in model class, everything covered in the course so far is on the table. You don't need to compare different ones, we will do that with the compiled results of all assignments.

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np


Load and investigate the data

In [2]:
df = pd.read_csv("CASP.csv")
df.head()

,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = df.corr()

plt.figure(figsize=(10,8))
sns.heatmap(
    corr,
    annot=False,
    cmap="coolwarm",
    square=True
)

plt.title("Correlation matrix")
plt.show()

Build feature matrix and target vector. Add scaling if needed for your model.

In [3]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Choose a Regression model, build, train and optimise

In [4]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

grid_enet_params = {
    "alpha": [0.001, 0.01, 0.1, 1],
    "l1_ratio": [0.2, 0.5, 0.8]
}

model_enet = ElasticNet(max_iter=10000)

grid_enet = GridSearchCV(
    model_enet,
    grid_enet_params,
    scoring="r2"
)

grid_enet.fit(X_train, y_train)

print(f"best parameters: {grid_enet.best_params_}, best score: {grid_enet.best_score_}")


best parameters: {'alpha': 0.001, 'l1_ratio': 0.8}, best score: 0.8354692129949104


In [5]:
from sklearn.metrics import r2_score, mean_squared_error

best_enet = ElasticNet(
    alpha=grid_enet.best_params_["alpha"],
    l1_ratio=grid_enet.best_params_["l1_ratio"],
    max_iter=10000
)

best_enet.fit(X_train, y_train)

y_pred_enet = best_enet.predict(X_test)

print(f"R^2: {r2_score(y_test, y_pred_enet)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_enet)}")


R^2: 0.8365984459118544
MSE: 5.843358637476808


In [ ]:
from sklearn.ensemble import RandomForestRegressor

grid_rf_params = {
    "n_estimators": [100, 150, 200],
    "max_depth": [None, 20, 30],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2, 3]
}

model_rf = RandomForestRegressor(random_state=42)

grid_rf = GridSearchCV(
    model_rf,
    grid_rf_params,
    scoring="r2"
)

grid_rf.fit(X_train, y_train)

print(f"best parameters: {grid_rf.best_params_}, best score: {grid_rf.best_score_}")


Evaluate your best model (MSE). Take note of training and test time (approximate).

In [ ]:
best_rf = RandomForestRegressor(
    n_estimators=grid_rf.best_params_["n_estimators"],
    max_depth=grid_rf.best_params_["max_depth"],
    max_features=grid_rf.best_params_["max_features"],
    min_samples_split=grid_rf.best_params_["min_samples_split"],
    min_samples_leaf=grid_rf.best_params_["min_samples_leaf"],
    random_state=42
)

best_rf.fit(X_train, y_train)

y_pred_rf = best_rf.predict(X_test)

print(f"R^2: {r2_score(y_test, y_pred_rf)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_rf)}")


#### Discussion points
1) Discuss your choice of model class.
2) How did you optimise your model? How did the best model perform?
3) How much time was needed for training the model and evaluations (approximation is enough)?
4) What limitations or shortcomings did you identify? What would be ideas to remedy or circumvent them?
5) In all its abstraction, what do the predictions of your model tell you?

1. I tried an Elastic Net Regression at first because it works well for highly correlated data (BME338 course). It performed poorly though, probably because of strong non-linearity. I then chose a Random Forest approach. Random Forest approaches often work very well for small to medium datasets and are very easy to train. To automate the parameter tuning, a grid search was used.
2. I used GridSearch for the tuning to automate the process. The best model performed:
3. The model training itself did not take very long but the parameter tuning took very long. It would probably take a lot less time but is not automated. 
4. The main shortcomings were in the Elastic Net because I used it because of the high correlations but it is not useful for non-linear data. The RF itself did not have many shortcomings. The only ones I noticed were the many parameters that could be tuned which leads to longer optimisation time.
5. Our model tells us how close we can predict the structure of a protein compared to the real one. We do not predict the structure itself but from the data we have we predict the error of maybe another model. E.g. We would have the error values from AlphaFold and what data it received to predict the structure. Out model thus predicts the possible error for the actual structure prediction model. In practice, this could lead to lower computation costs because if we predict a very high error for a protein prediction, it doesn't really make sense to run a computationally expensive model to get a bad result -> pre-testing